# 90s clips, both scenes, BoT-SORT, detected at 10 FPS

Detection and tracking run on **every 3rd frame** of the native ~30 fps footage, i.e. an
effective 10 FPS. This is the standing FPS principle for this project: full-rate processing
isn't required, and a real-time deployment wouldn't run at 30 fps either.

## The one thing that must not be got wrong: the time base

The dashboard plays the **original 30 fps clip** and draws overlays from this run's CSV, so
detections have to remain addressable on the 30 fps timeline. Every row therefore records
its **original frame index** (0, 3, 6, 9, ...) and `time_s = original_frame / native_fps` -
NOT a renumbered 0,1,2,3 sequence.

Renumbering would be the easy mistake and it would silently play the analytics back at 3x
speed against the video: frame 300 of a re-indexed sequence is 10s in at 30 fps but is
actually 30s of real footage. Keeping true indices means the frontend can hold each
detection for the 3 frames until the next sample, and everything stays in sync.

## Tracker tuning for the lower rate

At 10 FPS a vehicle moves ~3x further between consecutive observations, which makes motion
association genuinely harder - this is a real cost of subsampling, not a free win. Two
parameters are adjusted to match:
  * `frame_rate=10` so BoT-SORT's Kalman motion model uses the correct time step;
  * `track_buffer=30` = 3s of tolerance at 10 FPS (the 10s runs used 60 frames = 2s at
    30 FPS), keeping a slightly longer memory to offset the harder association.

The annotated video is still written at the **native frame rate** with the most recent boxes
held between samples, so it stays in step with the source clip rather than playing 3x fast.

In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

# Pin numpy/scipy before any install, same fix as the earlier notebooks: boxmot pulls a
# newer numpy than scipy's compiled internals expect and breaks `import supervision`.
import numpy
PINS = f'"numpy=={numpy.__version__}"'
try:
    import scipy
    PINS += f' "scipy=={scipy.__version__}"'
except ImportError:
    pass
print("Pinning:", PINS)

In [ ]:
!pip install -q ultralytics sahi huggingface_hub supervision {PINS}
!pip install -q --no-deps "boxmot==22.0.0"
!pip install -q filterpy lapx scikit-learn timm click rich {PINS}

In [ ]:
import glob

SOURCES = ["multiroad", "intersection"]
DURATION = "90s"
MODEL_NAME = "visdrone-yolov11s"
MODEL_REPO = "dronefreak/visdrone-yolov11s"
FRAME_STRIDE = 3          # detect every 3rd frame => ~10 FPS from ~30 fps source
TARGET_FPS = 10.0

video_paths = {}
for source in SOURCES:
    found = glob.glob(f"/kaggle/input/**/{source}_{DURATION}.MP4", recursive=True)
    assert found, f"No {source}_{DURATION}.MP4 found under /kaggle/input"
    video_paths[source] = found[0]
    print(source, "->", found[0])

In [ ]:
import colorsys

VISDRONE_NAMES = {0: "pedestrian", 1: "people", 2: "bicycle", 3: "car", 4: "van",
                   5: "truck", 6: "tricycle", 7: "awning-tricycle", 8: "bus", 9: "motor", 10: "others"}
VISDRONE_TO_TAXONOMY = {
    "car": "car", "van": "LGV_candidate", "truck": "HGV_candidate", "bus": "bus",
    "motor": "motorcycle", "pedestrian": "pedestrian", "people": "pedestrian",
    "bicycle": "other", "tricycle": "other", "awning-tricycle": "other", "others": "other",
}


def track_color(track_id: int):
    """Deterministic per-track BGR colour - identical to pipeline/trajectory_trails.py."""
    hue = (track_id * 0.618033988749895) % 1.0
    r, g, b = colorsys.hsv_to_rgb(hue, 0.85, 1.0)
    return int(b * 255), int(g * 255), int(r * 255)

In [ ]:
import csv
import json
import time

import cv2
import numpy as np
from huggingface_hub import hf_hub_download
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

weights_path = hf_hub_download(repo_id=MODEL_REPO, filename="best.pt")
CONFIDENCE_THRESHOLD = 0.35
ROOT_OUTPUT_DIR = "/kaggle/working/level4_90s"


def make_botsort():
    import gdown
    import boxmot
    from boxmot.trackers.bbox.botsort import BotSort
    from boxmot.reid.core.reid import ReID

    weights_dir = os.path.join(os.path.dirname(boxmot.__file__), "..", "models")
    os.makedirs(weights_dir, exist_ok=True)
    reid_weights = os.path.join(weights_dir, "osnet_x0_25_msmt17.pt")
    if not os.path.exists(reid_weights):
        gdown.download(id="1sSwXSUlj4_tHZequ_iZ8w_Jh0VaRQMqF", output=reid_weights, quiet=False)

    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    reid = ReID(path=reid_weights, device=device, half=torch.cuda.is_available())
    # frame_rate matches the ACTUAL observation rate (10), not the video's 30 - the Kalman
    # model would otherwise assume 3x smaller motion between updates than really occurs.
    return BotSort(reid_model=reid.model, track_buffer=30, frame_rate=int(TARGET_FPS),
                   with_reid=True)


for source in SOURCES:
    video_path = video_paths[source]
    output_name = f"{MODEL_NAME}-botsort"
    print(f"\n=== {source} / {output_name} / {DURATION} @ {TARGET_FPS:.0f}fps ===")
    model_dir = f"{ROOT_OUTPUT_DIR}/{source}/{output_name}"
    os.makedirs(model_dir, exist_ok=True)

    detection_model = AutoDetectionModel.from_pretrained(
        model_type="ultralytics", model_path=weights_path,
        confidence_threshold=CONFIDENCE_THRESHOLD,
        device="cuda:0" if torch.cuda.is_available() else "cpu",
    )
    tracker = make_botsort()

    cap = cv2.VideoCapture(video_path)
    native_fps = cap.get(cv2.CAP_PROP_FPS)
    w, h = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    n_frames_total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    # Annotated video written at NATIVE fps so it stays in step with the source clip.
    writer = cv2.VideoWriter(f"{model_dir}/video_raw.mp4",
                             cv2.VideoWriter_fourcc(*"mp4v"), native_fps, (w, h))

    rows = []
    last_track_rows = []      # held between detected frames so the video stays 30 fps
    t0 = time.time()
    frame_idx = 0
    n_detected = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        if frame_idx % FRAME_STRIDE == 0:
            result = get_sliced_prediction(
                frame, detection_model, slice_height=768, slice_width=768,
                overlap_height_ratio=0.2, overlap_width_ratio=0.2, verbose=0,
            )
            preds = result.object_prediction_list
            if preds:
                dets = np.array(
                    [[*p.bbox.to_xyxy(), p.score.value, p.category.id] for p in preds],
                    dtype=np.float32)
            else:
                dets = np.empty((0, 6), dtype=np.float32)
            tracked_arr = np.asarray(tracker.update(dets, frame))
            # boxmot columns: x1,y1,x2,y2,id,conf,cls,det_ind
            last_track_rows = [(r[:4], r[6], r[4], r[5]) for r in tracked_arr]
            n_detected += 1

            for xyxy_box, cid, tid, conf in last_track_rows:
                native = VISDRONE_NAMES.get(int(cid), "others")
                taxonomy = VISDRONE_TO_TAXONOMY.get(native, "other")
                x1, y1, x2, y2 = [float(v) for v in xyxy_box]
                rows.append({
                    # ORIGINAL frame index and true wall-clock time - see the markdown above.
                    "frame": frame_idx,
                    "time_s": round(frame_idx / native_fps, 3),
                    "track_id": int(tid), "native_class": native, "taxonomy_class": taxonomy,
                    "confidence": round(float(conf), 3),
                    "x1": round(x1, 1), "y1": round(y1, 1),
                    "x2": round(x2, 1), "y2": round(y2, 1),
                })

        annotated = frame.copy()
        for xyxy_box, cid, tid, conf in last_track_rows:
            native = VISDRONE_NAMES.get(int(cid), "others")
            taxonomy = VISDRONE_TO_TAXONOMY.get(native, "other")
            colour = track_color(int(tid))
            x1, y1, x2, y2 = [int(v) for v in xyxy_box]
            cv2.rectangle(annotated, (x1, y1), (x2, y2), colour, 2)
            cv2.putText(annotated, f"#{int(tid)} {taxonomy}", (x1, max(0, y1 - 5)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, colour, 1)
        writer.write(annotated)

        frame_idx += 1
        if frame_idx % 300 == 0:
            el = time.time() - t0
            print(f"  frame {frame_idx}/{n_frames_total} "
                  f"({n_detected} detected, {el:.0f}s elapsed, "
                  f"{el / max(1, n_detected):.2f}s per detected frame)")

    cap.release()
    writer.release()

    fieldnames = ["frame", "time_s", "track_id", "native_class", "taxonomy_class",
                  "confidence", "x1", "y1", "x2", "y2"]
    with open(f"{model_dir}/detections.csv", "w", newline="") as f:
        wcsv = csv.DictWriter(f, fieldnames=fieldnames)
        wcsv.writeheader()
        wcsv.writerows(rows)

    n_tracks = len(set(r["track_id"] for r in rows))
    meta = {
        "source_video": source,
        "duration_s": int(DURATION.rstrip("s")),
        "confidence_threshold": CONFIDENCE_THRESHOLD,
        "model": output_name,
        "weights_source": MODEL_REPO,
        "frames_processed": frame_idx,
        "frames_detected": n_detected,
        "frame_stride": FRAME_STRIDE,
        "detection_fps": round(native_fps / FRAME_STRIDE, 2),
        # `fps` stays the NATIVE rate: it is what the frontend uses to convert playback time
        # to a frame index, and the frame numbers in detections.csv are native-timeline.
        "fps": native_fps,
        "frame_width": w,
        "frame_height": h,
        "detection_rows": len(rows),
        "unique_tracks": n_tracks,
        "tracker": f"boxmot.BotSort (+ReID, track_buffer=30, frame_rate={int(TARGET_FPS)})",
        "sliced_inference": {"slice_size": 768, "overlap": 0.2, "tool": "SAHI"},
    }
    with open(f"{model_dir}/meta.json", "w") as f:
        json.dump(meta, f, indent=2)

    print(f"{source}: {frame_idx} frames ({n_detected} detected), {len(rows)} rows, "
          f"{n_tracks} tracks, {time.time() - t0:.0f}s")

In [ ]:
# Browser-compatible H.264, native frame rate preserved.
for source in SOURCES:
    model_dir = f"{ROOT_OUTPUT_DIR}/{source}/{MODEL_NAME}-botsort"
    src = f"{model_dir}/video_raw.mp4"
    dst = f"{model_dir}/video_h264.mp4"
    os.system(f'ffmpeg -y -i "{src}" -c:v libx264 -pix_fmt yuv420p -movflags +faststart "{dst}"')
    if os.path.exists(dst):
        os.remove(src)      # raw mp4v copy is large and redundant once transcoded
    print(source, "->", dst, os.path.exists(dst))

In [ ]:
# Sanity check the time base before pulling anything down: frame indices must be strided on
# the native timeline, and time_s must equal frame/native_fps.
import pandas as pd
for source in SOURCES:
    d = f"{ROOT_OUTPUT_DIR}/{source}/{MODEL_NAME}-botsort"
    df = pd.read_csv(f"{d}/detections.csv")
    m = json.load(open(f"{d}/meta.json"))
    frames = sorted(df["frame"].unique())
    steps = set(np.diff(frames))
    err = (df["time_s"] - df["frame"] / m["fps"]).abs().max()
    print(f"{source}: {len(frames)} detected frames, first={frames[:4]}, "
          f"stride set={steps}, max |time_s - frame/fps| = {err:.4f}")
    print(f"  native fps {m['fps']:.3f}, detection fps {m['detection_fps']}, "
          f"span {frames[-1] / m['fps']:.1f}s")
    assert steps <= {FRAME_STRIDE}, f"unexpected frame stride: {steps}"
    assert err < 0.01, "time_s does not match frame/native_fps"
print("\nTime base OK - detections sit on the native 30fps timeline.")